# Murf Finch speech-to-text on Amazon SageMaker

This notebook deploys a Murf Finch model package from AWS Marketplace and shows how to call it:

1. Real-time transcription (`InvokeEndpoint`): send an audio file, get the transcript.
2. Language and word timestamps.
3. Live streaming (`InvokeEndpointWithBidirectionalStream`): send audio as it is captured, get transcripts per turn.
4. Batch transform: transcribe a folder of audio files from Amazon S3.

**Before you start**

- Subscribe to the listing in AWS Marketplace and pick a version: Finch Global (English and European languages) or Finch Indic (Indian languages).
- Make sure the account has quota for the instance type below (Service Quotas, "ml.g5.2xlarge for endpoint usage", and "ml.g6.2xlarge for transform job usage" for batch).
- Run this from SageMaker Studio or a notebook instance whose execution role has `AmazonSageMakerFullAccess`.

The container runs with network isolation: audio never leaves your account.

In [ ]:
import json, time, urllib.request
import boto3, sagemaker

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
sm = boto3.client("sagemaker", region_name=region)
runtime = boto3.client("sagemaker-runtime", region_name=region)

VERSION = "global"   # "global" (English and European languages) or "indic" (Indian languages)

# Model package ARNs per Region. If your Region is missing, paste the ARN from the
# listing's "Continue to configuration" page into MODEL_PACKAGE_ARN.
MODEL_PACKAGE_ARNS = {"global": {}, "indic": {}}
MODEL_PACKAGE_ARN = MODEL_PACKAGE_ARNS[VERSION].get(region, "<model-package-arn-for-your-region>")
INSTANCE_TYPE = "ml.g5.2xlarge"         # real-time; see the README for other supported types
BATCH_INSTANCE_TYPE = "ml.g6.2xlarge"   # batch transform: ml.g6.2xlarge, ml.g6.xlarge or ml.g5.xlarge
name = f"murf-finch-{time.strftime('%Y%m%d-%H%M%S')}"

# Sample clips from the FLEURS dataset (CC BY 4.0).
SAMPLE_EN = urllib.request.urlopen("https://raw.githubusercontent.com/murf-ai/murf-finch-sagemaker/main/samples/en_us_sample.wav").read()
SAMPLE_HI = urllib.request.urlopen("https://raw.githubusercontent.com/murf-ai/murf-finch-sagemaker/main/samples/hi_in_sample.wav").read()

## Deploy a real-time endpoint

Creating the endpoint takes about 10 minutes.

In [ ]:
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
        "ContainerStartupHealthCheckTimeoutInSeconds": 900,
        "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 80})
print("endpoint", name, "is InService")

## 1. Real-time transcription

In [ ]:
t0 = time.perf_counter()
resp = runtime.invoke_endpoint(EndpointName=name, ContentType="audio/wav", Body=SAMPLE_EN)
result = json.loads(resp["Body"].read())
print(f"{result['audio_duration_secs']:.2f} s of audio in {time.perf_counter() - t0:.2f} s")
print(result["language_code"], ":", result["text"])

## 2. Language and word timestamps

Set options in `CustomAttributes`. Omit `language_code` to detect the language automatically. Finch Indic returns word start and end times; Finch Global returns an empty `words` list.

In [ ]:
sample, lang = (SAMPLE_HI, "hi") if VERSION == "indic" else (SAMPLE_EN, "en")
resp = runtime.invoke_endpoint(EndpointName=name, ContentType="audio/wav", Body=sample,
                               CustomAttributes=f"language_code={lang},timestamps_granularity=word")
result = json.loads(resp["Body"].read())
print(result["text"])
for w in result["words"]:
    print(f"{w['start']:5.2f}-{w['end']:5.2f}  {w['text']}")

## 3. Live streaming

Streaming uses SageMaker's HTTP/2 bidirectional API, which needs Python 3.12+ and the `aws-sdk-sagemaker-runtime-http2` package (boto3 does not support it yet). The next cell downloads [examples/stream.py](https://github.com/murf-ai/murf-finch-sagemaker/blob/main/examples/stream.py) and streams the sample at real-time pace. You get `partial` transcripts while the speaker talks and one `final` per turn.

In [ ]:
import subprocess, sys
assert sys.version_info >= (3, 12), "streaming needs a Python 3.12+ kernel"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aws-sdk-sagemaker-runtime-http2[awscrt]>=0.11"], check=True)
base = "https://raw.githubusercontent.com/murf-ai/murf-finch-sagemaker/main"
open("stream.py", "wb").write(urllib.request.urlopen(f"{base}/examples/stream.py").read())
open("sample.wav", "wb").write(SAMPLE_HI if VERSION == "indic" else SAMPLE_EN)
print(subprocess.run([sys.executable, "stream.py", "--endpoint", name, "--region", region, "sample.wav"],
                     capture_output=True, text=True).stdout)

## 4. Batch transform

Each audio file in the input prefix is transcribed; the output prefix gets one JSON transcript per file.

In [ ]:
s3 = boto3.client("s3", region_name=region)
prefix = f"murf-finch-batch/{name}"
s3.put_object(Bucket=bucket, Key=f"{prefix}/input/en_us_sample.wav", Body=SAMPLE_EN)
s3.put_object(Bucket=bucket, Key=f"{prefix}/input/hi_in_sample.wav", Body=SAMPLE_HI)

sm.create_transform_job(
    TransformJobName=name,
    ModelName=name,
    MaxConcurrentTransforms=4,
    BatchStrategy="SingleRecord",
    MaxPayloadInMB=100,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{prefix}/input/"}},
        "ContentType": "audio/wav",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": f"s3://{bucket}/{prefix}/output/", "Accept": "application/json"},
    TransformResources={"InstanceType": BATCH_INSTANCE_TYPE, "InstanceCount": 1},
)
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=name)
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/output/").get("Contents", []):
    out = json.loads(s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read())
    print(obj["Key"].rsplit("/", 1)[-1], "->", out["text"])

## Clean up

The endpoint is billed while it runs. Delete it when you are done.

In [ ]:
sm.delete_endpoint(EndpointName=name)
sm.delete_endpoint_config(EndpointConfigName=name)
sm.delete_model(ModelName=name)